# 07. 규칙 적용성 검증 및 후보별 근거 감사

06번에서는 속성별 지식 검색을 적용했지만 Top-1 예측은 바뀌지 않았습니다. 이는 근거 점수의 크기만 조정할 문제가 아니라, 검색된 규칙이 해당 제품과 후보 코드에 실제로 적용되는지 확인해야 한다는 뜻입니다.

이 노트북은 자동 재정렬 성능을 억지로 높이지 않습니다. 후보별 규칙 근거를 유형별로 분리하고, 제품-규칙 적용성을 사람이 검토할 수 있는 annotation worksheet를 생성합니다. 정답 HS 코드는 후보 생성이나 근거 검색에 사용하지 않고 평가에만 사용합니다.


In [ ]:
import re, hashlib
import numpy as np
import pandas as pd
from urllib.parse import quote
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score


In [ ]:
BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
cases_file = quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv',safe='')
data_all = pd.read_csv(BASE+cases_file).fillna('')
kb = pd.read_json(BASE+'hs_knowledge_base.jsonl',lines=True).fillna('')
input_cols = ['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
data_all['model_text'] = data_all[input_cols].astype(str).agg(' '.join,axis=1)
data_all['duplicate_group'] = data_all[input_cols].astype(str).agg('||'.join,axis=1).map(lambda x:hashlib.md5(x.encode()).hexdigest())
eligible = data_all.groupby('HS Code').filter(lambda x:len(x)>=4)
group_counts = eligible.groupby('HS Code')['duplicate_group'].nunique()
labels = sorted(group_counts[group_counts>=4].index.astype(str))
data = eligible[eligible['HS Code'].astype(str).isin(labels)].copy().reset_index(drop=True)
data['HS Code'] = data['HS Code'].astype(str)
print('제품:',len(data),'| HS 코드:',data['HS Code'].nunique(),'| 지식 문서:',len(kb))


## 1. 지식 문서의 규칙 유형 감사

규칙 유형은 원본 `rule_type`을 우선합니다. `제외조항` 시트에 있다는 이유만으로 모든 문서를 제외 규칙으로 취급하지 않습니다. 특히 `조건부 포함`은 조건 충족 여부가 확인되기 전까지 긍정 또는 제외 점수에 자동 반영하지 않습니다.


In [ ]:
kb['rule_type_clean'] = kb['rule_type'].astype(str).str.strip()
positive_types = {'포함','소호별 포함 범위','분류원칙·예외'}
exclusion_types = {'제외','제외·예외'}
conditional_types = {'조건부 포함','조건부 분류'}
mixed_types = {'소호별 포함 범위 및 제외'}
def classify_rule(value):
    if value in exclusion_types: return 'exclusion'
    if value in mixed_types: return 'mixed'
    if value in conditional_types: return 'conditional'
    if value in positive_types: return 'positive'
    return 'unclassified'
kb['rule_category'] = kb['rule_type_clean'].map(classify_rule)
kb['hs_code_clean'] = kb['hs_code'].astype(str).str.strip()
display(kb.groupby(['source_sheet','rule_type_clean','rule_category']).size().rename('n').reset_index())
display(kb[kb['rule_category'].eq('unclassified')][['knowledge_id','source_sheet','hs_code','rule_type','document_text']])


## 2. 후보 생성과 지식 검색

각 fold에서 SVM은 학습 데이터만 사용해 Top-3 후보를 생성합니다. 지식 문서는 독립된 KB 전체에서 검색합니다. HS 코드가 명시된 문서는 후보 코드와 계층적으로 호환되는 경우만 후보별 근거로 연결하고, 코드가 비어 있는 일반 규정은 `general`로 표시해 코드별 규칙과 구분합니다.


In [ ]:
def code_digits(value): return re.sub(r'\\D','',str(value))
def code_compatible(rule_code,candidate_code):
    a,b = code_digits(rule_code),code_digits(candidate_code)
    return bool(a and b and (a.startswith(b) or b.startswith(a)))
kb['search_text'] = (kb['document_text'].astype(str)+' '+kb['rule_type_clean'].astype(str)+' '+kb['source_sheet'].astype(str)).str.strip()
kb_vectorizer = FeatureUnion([('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),max_features=30000,sublinear_tf=True)),('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),max_features=30000,sublinear_tf=True))])
kb_matrix = kb_vectorizer.fit_transform(kb['search_text'])
def field_similarity(value):
    value = str(value).strip()
    if not value: return np.zeros(len(kb))
    return (kb_matrix @ kb_vectorizer.transform([value]).T).toarray().ravel()
N_SPLITS = 4
cv = StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=42)
candidate_rows, evidence_rows, fold_rows = [],[],[]
for fold,(tr,te) in enumerate(cv.split(data['model_text'],data['HS Code'],groups=data['duplicate_group']),1):
    train,test = data.iloc[tr],data.iloc[te]
    vec = FeatureUnion([('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1,max_features=30000,sublinear_tf=True)),('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1,max_features=30000,sublinear_tf=True))])
    Xtr,Xte = vec.fit_transform(train['model_text']),vec.transform(test['model_text'])
    clf = LinearSVC(C=1,class_weight='balanced',max_iter=5000).fit(Xtr,train['HS Code'])
    decision = clf.decision_function(Xte)
    if decision.ndim == 1: decision = np.column_stack([-decision,decision])
    for local_i,(_,row) in enumerate(test.iterrows()):
        order = np.argsort(decision[local_i])[::-1][:3]
        for rank,j in enumerate(order,1):
            candidate = str(clf.classes_[j])
            candidate_rows.append({'fold':fold,'row_index':int(row.name),'true_hs':row['HS Code'],'candidate_rank':rank,'candidate_hs':candidate,'svm_decision_score':float(decision[local_i,j])})
            found = {}
            for field in input_cols:
                sims = field_similarity(row[field])
                compatible = kb.apply(lambda d: (not str(d['hs_code_clean']).strip()) or code_compatible(d['hs_code_clean'],candidate),axis=1).to_numpy()
                idxs = np.where(compatible)[0]
                if len(idxs)==0: continue
                top = idxs[np.argsort(sims[idxs])[::-1][:5]]
                for kb_idx in top:
                    d = kb.iloc[kb_idx]
                    key = str(d['knowledge_id'])
                    item = {'fold':fold,'row_index':int(row.name),'candidate_hs':candidate,'candidate_rank':rank,'field':field,'field_similarity':float(sims[kb_idx]),'knowledge_id':key,'source_sheet':d['source_sheet'],'rule_hs_code':d['hs_code_clean'],'rule_type':d['rule_type_clean'],'rule_category':d['rule_category'],'code_scope':'general' if not d['hs_code_clean'] else 'code_specific','document_text':d['document_text']}
                    if key not in found or item['field_similarity']>found[key]['field_similarity']: found[key]=item
            evidence_rows.extend(found.values())
    fold_rows.append({'fold':fold,'train_n':len(tr),'test_n':len(te),'group_overlap':len(set(train['duplicate_group'])&set(test['duplicate_group']))})
    print('fold',fold,'완료')
candidates = pd.DataFrame(candidate_rows)
evidence = pd.DataFrame(evidence_rows)
fold_audit = pd.DataFrame(fold_rows)
display(fold_audit)
print('후보 행:',len(candidates),'| 후보-근거 연결:',len(evidence))


## 3. 후보별 근거 구성 진단

여기서 보는 것은 규칙 적용성의 정답이 아니라 검색 결과의 구성입니다. 문서 유형별 빈도, 일반 규정과 코드별 규정의 비중, 후보 순위별 근거 수를 확인합니다. 검색 유사도가 높다는 사실만으로 규칙이 제품에 적용된다고 해석하면 안 됩니다.


In [ ]:
evidence_summary = evidence.groupby(['rule_category','code_scope']).size().rename('n').reset_index()
rank_summary = evidence.groupby(['candidate_rank','rule_category']).size().rename('n').reset_index()
display(evidence_summary)
display(rank_summary)
display(evidence.groupby(['field','rule_category']).size().rename('n').reset_index())
print('일반 규정 비중:',round(evidence['code_scope'].eq('general').mean(),3) if len(evidence) else np.nan)


## 4. 수작업 적용성 검토표 생성

HS 코드별 최대 4개 제품을 표본 추출하고, 각 제품의 Top-3 후보마다 유사도 상위 근거 3개를 검토표로 만듭니다. 검토자는 `applicability`에 `적용`, `미적용`, `불명확` 중 하나를 입력하고, 조건부 규칙은 `condition_status`에 `충족`, `미충족`, `확인불가`를 기록합니다. 이 annotation 결과가 쌓이기 전에는 자동 규칙 점수로 최종 코드를 바꾸지 않습니다.


In [ ]:
sample_idx = data.groupby('HS Code',group_keys=False).apply(lambda x:x.sample(n=min(4,len(x)),random_state=42),include_groups=False).index
sample_products = set(sample_idx)
review_rows = []
for (fold,row_index,candidate_hs),g in evidence[evidence['row_index'].isin(sample_products)].groupby(['fold','row_index','candidate_hs']):
    top = g.sort_values('field_similarity',ascending=False).drop_duplicates('knowledge_id').head(3)
    for _,r in top.iterrows():
        review_rows.append({**r.to_dict(),'applicability':'','condition_status':'','reviewer_note':''})
review = pd.DataFrame(review_rows)
display(review.head(10))
print('수작업 검토 행:',len(review),'| 제품 수:',review['row_index'].nunique() if len(review) else 0)


## 5. 기준선 성능 및 후보 커버리지

이 단계의 목적은 규칙 근거를 감사하고 검토 데이터를 만드는 것입니다. 따라서 SVM 성능과 Top-3 후보 커버리지를 함께 기록하되, 검토자가 적용성을 라벨링하기 전에는 재정렬 성능을 주장하지 않습니다.


In [ ]:
top1 = candidates[candidates['candidate_rank'].eq(1)].copy()
top3 = candidates.groupby(['fold','row_index','true_hs'])['candidate_hs'].apply(list).reset_index(name='top3')
metrics = pd.DataFrame([{'n_products':len(top1),'accuracy':accuracy_score(top1['true_hs'],top1['candidate_hs']),'macro_f1':f1_score(top1['true_hs'],top1['candidate_hs'],average='macro',zero_division=0),'weighted_f1':f1_score(top1['true_hs'],top1['candidate_hs'],average='weighted',zero_division=0),'top3_hit_rate':top3.apply(lambda r:r['true_hs'] in r['top3'],axis=1).mean()}])
display(metrics)


## 6. 결과 저장

`07_candidate_predictions.csv`: fold별 Top-3 후보와 SVM 점수

`07_candidate_rule_evidence.csv`: 후보별 검색 근거

`07_manual_rule_review.csv`: 사람이 적용성을 검토할 annotation worksheet

`07_rule_evidence_summary.csv`: 근거 유형 및 범위별 빈도

`07_fold_audit.csv`: fold별 그룹 중복 점검


In [ ]:
candidates.to_csv('07_candidate_predictions.csv',index=False,encoding='utf-8-sig')
evidence.to_csv('07_candidate_rule_evidence.csv',index=False,encoding='utf-8-sig')
review.to_csv('07_manual_rule_review.csv',index=False,encoding='utf-8-sig')
evidence_summary.to_csv('07_rule_evidence_summary.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv('07_fold_audit.csv',index=False,encoding='utf-8-sig')
metrics.to_csv('07_baseline_metrics.csv',index=False,encoding='utf-8-sig')
print('07번 결과 저장 완료')
